This tutorial demonstrates how to build a player identification model for Go using PyTorch. We will cover data preparation, model definition, training, and inference. The model will identify Go players based on game records stored in CSV files. 

PyTorch is a powerful tool for handling large datasets and building deep learning models. By the end of this tutorial, you will have a solid understanding of how to implement a player identification model for Go. It is advised to run this tutorial on Ubuntu with a GPU for optimal performance.

**This tutorial is for reference purposes only. You are welcome to modify and improve the code as needed**. There is no restriction on the model architecture in this competition. Also, you can use any other libraries or frameworks you prefer.

In [ ]:
import json
import os

from tqdm import tqdm
import pandas as pd
import numpy as np

import torch
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, IterableDataset

from utils import SGFParsePlayerIdentification
from network import GoPlayerResNet

In [ ]:
import platform
print(platform.python_version())

In [ ]:
torch.__version__

# 0. Quick Look to the Dataset

Change the filepath to your own dataset when you run the code.

In [ ]:
TRAIN_PATH = './dataset/player_identification_train.csv'
train_data = pd.read_csv(TRAIN_PATH)
train_data

Here, each row in the dataset corresponds to a player in a match. The columns are as follows:
- `game_id`: Unique identifier for each match.
- `player_id`: Unique identifier for each player.
- `sgf_content`: The SGF content of the match.
- `color`: The color (black or white) of the player.

In [ ]:
train_data.info()

In [ ]:
unique_players = set(train_data['player_id'])
print(f"Total unique players: {len(unique_players)}")

In [ ]:
# Save selected players to a json file for future reference
players_dict = {i: player for i, player in enumerate(unique_players)}
json_path = f"./players_i2n.json"
with open(json_path, 'w') as f:
    json.dump(players_dict, f, indent=4)

# Load player ID to name mapping
ID_TO_PLAYER_NAME = json.load(open(json_path, 'r'))
# Create reverse mapping
PLAYER_NAME_TO_ID = {v: k for k, v in ID_TO_PLAYER_NAME.items()}

# Initialize feature extractor
sgf_parser = SGFParsePlayerIdentification()

FEATURES_DIR = f"./player-features/"
os.makedirs(FEATURES_DIR, exist_ok=True)

# 1. Feature Extraction

This section focuses on extracting relevant features from the SGF files to prepare the training dataset.

This section is **one time use** to extract features for training the model.

This process may take some time (approx. an hour) depending on your computer's performance. If you have already extracted the features, you can skip this section and move directly to the model training section.

Different from the rank prediction tutorial, in this tutorial, we will use a different approach for training: we extract features as 8 history length for each move, and train a model with these features. The loss function is selected as Triplet Loss, which is suitable for player identification tasks. In this way, the model learns to differentiate between players based on their unique playing styles and patterns over a series of moves. Therefore, after we extract features for each move, we will have randomly picked multiple training samples for every step in training. It means we don't set a fixed number of epochs for training, but rather a fixed number of steps. Also, we don't need to split the dataset into training and validation sets, since we will monitor the training process with the loss value.

In [ ]:
# Create a mapping from player name to list of their game indices
player_to_game_indices = train_data.groupby('player_id').indices

Below is the code to extract features and save them into npy format.

It might take some time to run this code (approx 2 hours). You can see the progress in the output logs.

**WARNING: The extracted features will take up a lot of disk space (approx 160GB). Make sure you have enough space before running this code.**

In [ ]:
player_to_game_to_move_count = sgf_parser.generate_training_features(
    data=train_data,
    player_to_game_indices=player_to_game_indices,
    player_name_to_id=PLAYER_NAME_TO_ID,
    features_dir=FEATURES_DIR,
)

In [ ]:
# save move counts to JSON
with open("./player_game_move_counts.json", 'w') as f:
    json.dump(player_to_game_to_move_count, f, indent=4)

# 2. Data Loader

After the features are extracted and saved in npy format, we can create data loaders for training.

**If you have already extracted the features, you can skip Section 1 feature extraction and move directly to this section.**

In [ ]:
# CONFIGS
# Data loader config
BATCH_SIZE = 100  # Adjust based on your GPU memory

# Model training config
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CHECKPOINT_DIR = "./player-trained-models/"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Training hyperparameters
STEPS_PER_EPOCH = 500_000    # Number of batches per epoch
LEARNING_RATE = 1e-4      # Learning rate for AdamW optimizer
MARGIN = 0.5              # Margin for triplet loss
NUM_EPOCHS = 5          # Total training epochs
LOG_INTERVAL = 10_000         # Print loss every N steps

# Load player mapping
json_path = "./player_game_move_counts.json"
with open(json_path, 'r') as f:
    player_game_move_counts = json.load(f)

print(f"Device: {DEVICE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Margin: {MARGIN}")
print(f"Number of epochs: {NUM_EPOCHS}")
print(f"Steps per epoch: {STEPS_PER_EPOCH:,}")
print(f"Total training steps: {NUM_EPOCHS * STEPS_PER_EPOCH:,}")

In [ ]:
class GoFeatureDataset(IterableDataset):
    def __init__(self, feature_dir:str, player_game_move_counts:dict, steps:int=1):
        '''
        feature_dir: Directory where feature .npy files are stored. i.e., "./features/50ppl_500_games"
        player_game_move_counts: A dict mapping player_id to another dict mapping game_idx to move_count.
        '''
        self.feature_dir = feature_dir
        self.player_game_move_counts = player_game_move_counts
        self.list_of_players = list(player_game_move_counts.keys())
        self.batch_size = BATCH_SIZE * steps

    def _is_file_valid(self, filepath:str) -> bool:
        return os.path.isfile(filepath)

    def __iter__(self):
        for _ in range(self.batch_size):
            positive_player, negative_player = np.random.choice(self.list_of_players, 2, replace=False)
            anchor_game, positive_game = np.random.choice(
                list(self.player_game_move_counts[positive_player].keys()), 2, replace=False)
            anchor_move = np.random.choice(self.player_game_move_counts[positive_player][anchor_game])
            positive_move = np.random.choice(self.player_game_move_counts[positive_player][positive_game])
            
            negative_game = np.random.choice(list(self.player_game_move_counts[negative_player].keys()))
            negative_move = np.random.choice(self.player_game_move_counts[negative_player][negative_game])

            anchor_filepath = f"{self.feature_dir}/{positive_player}/{anchor_game}_{anchor_move}.npy"
            positive_filepath = f"{self.feature_dir}/{positive_player}/{positive_game}_{positive_move}.npy"
            negative_filepath = f"{self.feature_dir}/{negative_player}/{negative_game}_{negative_move}.npy"

            anchor_valid = self._is_file_valid(anchor_filepath)
            positive_valid = self._is_file_valid(positive_filepath)
            negative_valid = self._is_file_valid(negative_filepath)

            if anchor_valid and positive_valid and negative_valid:
                anchor = np.load(anchor_filepath)
                positive = np.load(positive_filepath)
                negative = np.load(negative_filepath)
                yield anchor, positive, negative
            else:
                yield None, None, None

In [ ]:
train_dataset = GoFeatureDataset(
    feature_dir=FEATURES_DIR,
    player_game_move_counts=player_game_move_counts,
    steps=STEPS_PER_EPOCH
)
data_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, pin_memory=True)

# 3. Model Training

Details of the model architecture are implemented in `network.py` file. We use the `GoPlayerRestNet` class from this file to define our model.

`GoPlayerRestNet` is a residual network designed for rank prediction in Go. It consists of an initial convolutional layer, followed by multiple residual blocks, and ends with fully connected layers as embedding vector output. Different approach than classification, this architecture allows the model to discriminate between players based on their unique playing styles and patterns over a series of moves. Then, we use Triplet Loss as the loss function to train the model. As for the reference and baseline in this competition, we keep the architecture relatively simple yet effective. You are welcome to modify the architecture as needed. There is no restriction on the model architecture in this competition.

In [ ]:
model = GoPlayerResNet().to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
criterion = nn.TripletMarginLoss(margin=MARGIN, p=2, reduction='mean')

In [ ]:
num_params = sum(p.numel() for p in model.parameters())
print(f"Model initialized: {num_params:,} parameters")
print(f"Optimizer: AdamW (lr={LEARNING_RATE})")

In [ ]:
# Check model and data loader compatibility
received_batch = next(iter(data_loader))
anchors, positives, negatives = received_batch
print(f"Received batch shapes - Anchors: {anchors.shape}, Positives: {positives.shape}, Negatives: {negatives.shape}")

anchors = anchors.to(DEVICE, dtype=torch.float32)
positives = positives.to(DEVICE, dtype=torch.float32)
negatives = negatives.to(DEVICE, dtype=torch.float32)

model.eval()
with torch.no_grad():
    outputs_anchor = model(anchors)
    outputs_positive = model(positives)
    outputs_negative = model(negatives)
    print(f"Output embeddings shapes - Anchor: {outputs_anchor.shape}, Positive: {outputs_positive.shape}, Negative: {outputs_negative.shape}")
    loss = criterion(outputs_anchor, outputs_positive, outputs_negative)
    print(f"Initial loss: {loss.item()}")

Above, the initial loss value should be around `MARGIN` value (0.5). You should train until the loss value gets below 0.1, lower is better.

In [ ]:
def train_epoch(model, data_loader, criterion, optimizer, device, epoch):
    # Train for one epoch
    model.train()
    running_loss = 0.0
    average_loss = 0.0
    for step, batch in enumerate(tqdm(data_loader, total=STEPS_PER_EPOCH, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")):
        anchors, positives, negatives = batch
        anchors = anchors.to(device, dtype=torch.float32)
        positives = positives.to(device, dtype=torch.float32)
        negatives = negatives.to(device, dtype=torch.float32)

        optimizer.zero_grad()
        outputs_anchor = model(anchors)
        outputs_positive = model(positives)
        outputs_negative = model(negatives)

        loss = criterion(outputs_anchor, outputs_positive, outputs_negative)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        average_loss += loss.item()
        if (step + 1) % LOG_INTERVAL == 0:
            avg_loss = running_loss / LOG_INTERVAL
            print(f"Step [{step+1}/{STEPS_PER_EPOCH}], Loss: {avg_loss:.4f}")
            running_loss = 0.0

    return average_loss / (step + 1)

In [ ]:
# Main training loop
best_loss = float('inf')

for epoch in range(NUM_EPOCHS):
    print(f'Starting epoch {epoch + 1}/{NUM_EPOCHS}...')
    # Train one epoch
    avg_loss = train_epoch(model, data_loader, criterion, optimizer, DEVICE, epoch)
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS} - Average Loss: {avg_loss:.4f}")
    
    # Save best model
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), f'{CHECKPOINT_DIR}best_model.pth')
        print(f"Saved best model (loss: {avg_loss:.4f})")
    
    # Save epoch checkpoint
    torch.save(model.state_dict(), f'{CHECKPOINT_DIR}epoch_{epoch+1}.pth')
    print("-----------------------------------------------------")

print("Training completed!")
print(f"Best loss: {best_loss:.4f}")

# 4. Inference with Testing Data On-the-Fly

For inference, we will load the trained model and use it to predict player identities based on the testing data.

Candidate dataset is also provided. We will use this dataset to get centeroids for each player in the candidate dataset. During inference, we will compare the centroids with the embedding vector of the target sgf files of the candidate players to determine the top-5 most likely players. 

This approach is zero-shot learning, as we do not retrain the model with the candidate dataset. The model has already learned to extract meaningful features from the training data, and we leverage this knowledge to create centroids for the candidate players. If you would like to further improve the performance, you can also consider fine-tuning the model with the candidate dataset (few-shot learning). This is totally up to you. For the reference and baseline in this competition, we will use zero-shot learning for inference.

Then, we will save the results in the required submission format.

In [ ]:
sgf_parser = SGFParsePlayerIdentification()

model_path = f'{CHECKPOINT_DIR}best_model.pth'
model = GoPlayerResNet().to(DEVICE)
model.load_state_dict(torch.load(model_path, map_location=DEVICE))

### Extract Centeroid Features from Candidate References

In [ ]:
CANDIDATE_PATH = './dataset/player_identification_candidates.csv'
candidates_data = pd.read_csv(CANDIDATE_PATH)
candidates_data

In [ ]:
candidate_players = candidates_data['player_id'].unique().tolist()
print(f"Total candidate players: {len(candidate_players)}")

In [ ]:
import pickle

# Extract candidate game embeddings
centroids = {}
for idx, player_id in enumerate(candidate_players):
    print(f"Processing player: {player_id}, {idx+1}/{len(candidate_players)}")
    candidate_games = candidates_data[candidates_data['player_id'] == player_id]['sgf_content'].values
    candidate_colors = candidates_data[candidates_data['player_id'] == player_id]['color'].values
    candidate_game_embeddings = []
    for game,color in zip(candidate_games, candidate_colors):
        game_features = sgf_parser.extract_features(game, color.lower())
        game_features = np.array(game_features)  # (N, 17, 19, 19)
        game_features = torch.tensor(game_features, dtype=torch.float32).to(DEVICE)
        with torch.no_grad():
            model.eval()
            game_embeddings = model(game_features)
        candidate_game_embeddings.append(game_embeddings)
    candidate_game_all_embeddings = torch.cat(candidate_game_embeddings, dim=0)
    candidate_game_embeddings_avg = torch.mean(candidate_game_all_embeddings, dim=0)
    centroids[player_id] = candidate_game_embeddings_avg.cpu().numpy()

# Save centroids to a pickle file
with open(f'./candidate-centroids.pkl', 'wb') as f:
    pickle.dump(centroids, f)
print(f"Centroids Saved!")

### Load Centeroids and Predict Top-5 Players for Each Target SGF

The following code demonstrates how to perform zero-shot inference on the testing data using the trained model with centroids.

The target games are averaged to create a single embedding representing the target player. The distance between this average embedding and the centroids is calculated using L2 distance. The top-5 closest distances are identified as the most likely players for the target game. 

In [ ]:
# Load Centeroids
with open('./candidate-centroids.pkl', 'rb') as f:
    centroids = pickle.load(f)

centroids_list = list(centroids.values())
centroids_list = torch.tensor(np.array(centroids_list))
centroids_players_list = list(centroids.keys())
centroids_list.shape

In [ ]:
TEST_PATH = './dataset/player_identification_test.csv'
testing_data = pd.read_csv(TEST_PATH)
testing_data

In [ ]:
# Retrieve all the columns from the testing data where the column name starts with "sgf_"
sgf_columns = [col for col in testing_data.columns if col.startswith('sgf_')]
sgf_colors = [col for col in testing_data.columns if col.startswith('color_')]

# Create submission DataFrame
submission_df = pd.DataFrame(columns=['question_id', 'top_1', 'top_2', 'top_3', 'top_4', 'top_5'])

# Predict row by row
for index, row in testing_data.iterrows():
    print(f"Processing test sample {index+1}/{len(testing_data)}")
    sgf_contents = [row[col] for col in sgf_columns if not pd.isna(row[col])]
    colors = [row[col].lower() for col in sgf_colors if not pd.isna(row[col])]
    row_name = row['question_id']

    games_top_k = []
    player_embeddings = []
    for sgf_content, color in zip(sgf_contents, colors):
        game_features = sgf_parser.extract_features(sgf_content, color)
        game_features = np.array(game_features)  # (N, 17, 19, 19)
        game_features = torch.tensor(game_features, dtype=torch.float32).to(DEVICE)
        with torch.no_grad():
            model.eval()
            game_embeddings = model(game_features)
            player_embeddings.append(game_embeddings)

    # Average embedding over all test games
    avg_embeddings = torch.mean(torch.cat(player_embeddings, dim=0), dim=0).cpu()
    distances = [torch.dist(centroid, avg_embeddings.unsqueeze(0), p=2).item() for centroid in centroids_list]
    distances = np.array(distances)
    top_k_indices = distances.argsort()[:5]  # Get top 5 closest centroids
    final_top5 = [centroids_players_list[i] for i in top_k_indices]

    # Append predictions to submission DataFrame
    submission_df.loc[len(submission_df)] = [
        row_name,
        final_top5[0],
        final_top5[1],
        final_top5[2],
        final_top5[3],
        final_top5[4]
    ]
submission_df.to_csv('./submission-player.csv', index=False)
print("Submission file created.")

# End of the tutorial

This concludes the player identification tutorial. The trained best model of this tutorial will be shared. You should be able to achieve around 0.30 score on the test set with this model.

The tutorial is prepared by Serkan Kavak, NDHU AI Lab. If you have any questions or need further assistance, feel free to open an issue on the project's GitHub repository. Good luck with your player identification model!